# 📊 Métricas de Evaluación y Validación en Machine Learning

## Introducción

Bienvenido al notebook de **Métricas de Evaluación y Técnicas de Validación**, donde aprenderás a evaluar modelos de Machine Learning de manera rigurosa y profesional.

Entrenar un modelo es solo la mitad del trabajo. La evaluación adecuada es **fundamental** para:
- Medir el rendimiento real del modelo
- Comparar diferentes algoritmos objetivamente  
- Detectar problemas como overfitting o underfitting
- Seleccionar hiperparámetros óptimos
- Garantizar que el modelo funcione bien con datos nuevos

Un modelo que logra 95% de accuracy puede parecer excelente, pero si las clases están desbalanceadas 95:5, ¡un clasificador trivial que siempre predice la clase mayoritaria obtendría ese mismo score! Por eso necesitamos métricas más sofisticadas.

### 🎯 Objetivos de Aprendizaje

Al completar este notebook, serás capaz de:

1. **Implementar** K-Fold Cross-Validation desde cero para estimación robusta
2. **Calcular** métricas de clasificación (Precision, Recall, F1, Specificity)
3. **Interpretar** la matriz de confusión y sus implicaciones
4. **Construir** curvas ROC y calcular el área bajo la curva (AUC)
5. **Analizar** curvas Precision-Recall para datasets desbalanceados
6. **Implementar** Grid Search para optimización de hiperparámetros
7. **Comprender** el trade-off entre diferentes métricas de evaluación

### 📚 Contexto

Las métricas de evaluación han evolucionado junto con el Machine Learning. La **curva ROC** fue desarrollada durante la Segunda Guerra Mundial para análisis de señales de radar. El **F1-Score** combina precisión y recall de manera armónica. **Cross-Validation**, popularizado por Geisser & Stone, es estándar en la práctica moderna de ML.

## Tabla de Contenidos

- [1 - Configuración del Entorno](#1)
- [2 - Teoría de Evaluación y Validación](#2)
  - [2.1 - Train/Test Split y Overfitting](#2.1)
  - [2.2 - K-Fold Cross-Validation](#2.2)
  - [2.3 - Matriz de Confusión](#2.3)
  - [2.4 - Métricas de Clasificación](#2.4)
  - [2.5 - Curva ROC y AUC](#2.5)
  - [2.6 - Curva Precision-Recall](#2.6)
- [3 - Implementación desde CERO](#3)
  - [3.1 - K-Fold Cross-Validation](#3.1)
  - [3.2 - Clase MetricasClasificacion](#3.2)
- [4 - Ejercicios GRADED](#4)
  - [Ejercicio 1 - Calcular Métricas de Clasificación](#ex01)
  - [Ejercicio 2 - Calcular AUC-ROC](#ex02)
- [5 - Aplicación Práctica](#5)
  - [5.1 - Evaluación Completa de un Modelo](#5.1)
  - [5.2 - Comparación de Modelos](#5.2)
  - [5.3 - Grid Search para Hiperparámetros](#5.3)
- [6 - Resumen y Conceptos Clave](#6)
- [7 - Referencias](#7)

<a name='1'></a>
## 1 - Configuración del Entorno

Importamos las bibliotecas necesarias y configuramos el entorno de trabajo.

In [ ]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
from pathlib import Path

# Agregar el directorio raíz al path de manera robusta
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Verificar que las utilidades se pueden importar
try:
    from utils.plot_utils import plot_confusion_matrix, plot_learning_curve
    from utils.testing_utils import print_success, print_error, print_info
    print("✅ Entorno configurado correctamente")
    print(f"📁 Raíz del proyecto: {project_root}")
except ImportError as e:
    print("❌ Error al importar utilidades")
    print("\n💡 Soluciones:")
    print("   1. Ejecuta 'pip install -e .' desde la raíz del proyecto")
    print("   2. O inicia Jupyter desde la raíz: cd ML-FROM-ZERO-PYTHON && jupyter notebook")
    print(f"\n🔍 Error detallado: {e}")
    raise

<a name='2'></a>
## 2 - Teoría de Evaluación y Validación

<a name='2.1'></a>
### 2.1 - Train/Test Split y Overfitting

El problema fundamental del aprendizaje supervisado es **generalización**: queremos que el modelo funcione bien en datos que **nunca ha visto**.

#### Overfitting vs Underfitting

- **Underfitting** (alto sesgo): El modelo es demasiado simple, no captura patrones importantes
  $$\text{Error}_{\text{train}} \text{ alto}, \quad \text{Error}_{\text{test}} \text{ alto}$$

- **Overfitting** (alta varianza): El modelo memoriza el training set, incluido el ruido
  $$\text{Error}_{\text{train}} \text{ bajo}, \quad \text{Error}_{\text{test}} \text{ alto}$$

- **Balance óptimo**: Generalización correcta
  $$\text{Error}_{\text{train}} \text{ bajo}, \quad \text{Error}_{\text{test}} \text{ bajo}$$

**Train/Test Split**: División típica 70-80% entrenamiento, 20-30% prueba.

<a name='2.2'></a>
### 2.2 - K-Fold Cross-Validation

**K-Fold Cross-Validation** maximiza el uso de los datos disponibles y proporciona estimaciones más robustas.

#### Procedimiento:

1. Dividir el dataset en $K$ subconjuntos (folds) de tamaño similar
2. Para cada fold $k = 1, 2, ..., K$:
   - Entrenar el modelo con $K-1$ folds
   - Evaluar en el fold $k$ (como conjunto de prueba)
3. Calcular la métrica promedio y desviación estándar:

$$\text{Score}_{\text{CV}} = \frac{1}{K} \sum_{k=1}^{K} \text{Score}_k$$

$$\sigma_{\text{CV}} = \sqrt{\frac{1}{K} \sum_{k=1}^{K} (\text{Score}_k - \text{Score}_{\text{CV}})^2}$$

#### Ventajas:
- Usa todos los datos para entrenamiento y evaluación
- Reduce varianza en la estimación del rendimiento
- Detecta mejor el overfitting

**Valor típico**: $K=5$ o $K=10$. Para datasets pequeños, usar **Leave-One-Out CV** ($K=n$).

<a name='2.3'></a>
### 2.3 - Matriz de Confusión

La **matriz de confusión** desglosa las predicciones en 4 categorías para clasificación binaria:

$$\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}$$

donde:
- **TP (True Positives)**: Correctamente predichos como positivos
- **TN (True Negatives)**: Correctamente predichos como negativos
- **FP (False Positives)**: Incorrectamente predichos como positivos (Error Tipo I)
- **FN (False Negatives)**: Incorrectamente predichos como negativos (Error Tipo II)

<a name='2.4'></a>
### 2.4 - Métricas de Clasificación

#### **1. Accuracy (Exactitud)**
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

- **Interpretación**: Proporción de predicciones correctas
- **Problema**: Engañosa con clases desbalanceadas

#### **2. Precision (Precisión)**
$$\text{Precision} = \frac{TP}{TP + FP}$$

- **Interpretación**: De los predichos como positivos, ¿cuántos son realmente positivos?
- **Uso**: Minimizar falsos positivos (ej: detección de spam)

#### **3. Recall (Sensibilidad, True Positive Rate)**
$$\text{Recall} = \frac{TP}{TP + FN}$$

- **Interpretación**: De los positivos reales, ¿cuántos detectamos?
- **Uso**: Minimizar falsos negativos (ej: diagnóstico médico de enfermedades graves)

#### **4. F1-Score**
$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \cdot TP}{2 \cdot TP + FP + FN}$$

- **Interpretación**: Media armónica de Precision y Recall
- **Uso**: Balance entre Precision y Recall

#### **5. Specificity (True Negative Rate)**
$$\text{Specificity} = \frac{TN}{TN + FP}$$

- **Interpretación**: De los negativos reales, ¿cuántos identificamos correctamente?

#### **Trade-offs**:
- **Precision ↑ → Recall ↓**: Ser más estricto reduce FP pero aumenta FN
- **Recall ↑ → Precision ↓**: Ser más permisivo reduce FN pero aumenta FP

<a name='2.5'></a>
### 2.5 - Curva ROC y AUC

La **curva ROC** (Receiver Operating Characteristic) visualiza el rendimiento del clasificador para diferentes umbrales de decisión.

#### Ejes:
- **Eje X**: False Positive Rate (FPR) = $1 - \text{Specificity} = \frac{FP}{FP + TN}$
- **Eje Y**: True Positive Rate (TPR) = Recall = $\frac{TP}{TP + FN}$

#### Construcción:
1. Para cada threshold $t \in [0, 1]$:
   - Clasificar: $\hat{y} = 1$ si $P(y=1|x) \geq t$, sino $\hat{y} = 0$
   - Calcular $(\text{FPR}_t, \text{TPR}_t)$
2. Graficar todos los puntos $(\text{FPR}_t, \text{TPR}_t)$

#### **AUC (Area Under the Curve)**
$$\text{AUC} = \int_{0}^{1} \text{TPR}(\text{FPR}) \, d(\text{FPR})$$

**Interpretación**:
- **AUC = 1.0**: Clasificador perfecto
- **AUC = 0.5**: Clasificador aleatorio (línea diagonal)
- **AUC < 0.5**: Peor que aleatorio (predicciones invertidas)
- **AUC > 0.9**: Excelente
- **0.7 ≤ AUC < 0.9**: Bueno

**Ventaja**: Independiente del threshold, robusta a desbalance de clases.

<a name='2.6'></a>
### 2.6 - Curva Precision-Recall

La **curva Precision-Recall** es especialmente útil cuando:
- Las clases están **muy desbalanceadas**
- Los falsos positivos son costosos

#### Ejes:
- **Eje X**: Recall = $\frac{TP}{TP + FN}$
- **Eje Y**: Precision = $\frac{TP}{TP + FP}$

#### Average Precision (AP)
$$\text{AP} = \sum_{k=1}^{n} \text{Precision}_k \cdot \Delta \text{Recall}_k$$

**Uso**: Detección de objetos, recuperación de información, sistemas de recomendación.

<a name='3'></a>
## 3 - Implementación desde CERO

<a name='3.1'></a>
### 3.1 - K-Fold Cross-Validation

Implementaremos K-Fold CV desde cero usando solo NumPy.

In [ ]:
class KFoldCV:
    """
    K-Fold Cross-Validation implementado desde cero.
    
    Parámetros
    ----------
    n_splits : int, opcional (default=5)
        Número de folds
    shuffle : bool, opcional (default=True)
        Si True, baraja los datos antes de dividir
    random_state : int, opcional (default=None)
        Semilla para reproducibilidad
    """
    
    def __init__(self, n_splits=5, shuffle=True, random_state=None):
        self.n_splits = n_splits
        self.shuffle = shuffle
        self.random_state = random_state
    
    def split(self, X, y=None):
        """
        Genera índices de train/test para cada fold.
        
        Parámetros
        ----------
        X : array-like
            Datos de entrada
        y : array-like, opcional
            Etiquetas (no se usa, incluido por compatibilidad)
        
        Yields
        ------
        train_idx : ndarray
            Índices para el conjunto de entrenamiento
        test_idx : ndarray
            Índices para el conjunto de prueba
        """
        n_samples = len(X)
        indices = np.arange(n_samples)
        
        # Barajar si es necesario
        if self.shuffle:
            if self.random_state is not None:
                np.random.seed(self.random_state)
            np.random.shuffle(indices)
        
        # Calcular tamaños de cada fold
        fold_sizes = np.full(self.n_splits, n_samples // self.n_splits, dtype=int)
        fold_sizes[:n_samples % self.n_splits] += 1
        
        # Generar cada fold
        current = 0
        for fold_size in fold_sizes:
            start, stop = current, current + fold_size
            test_idx = indices[start:stop]
            train_idx = np.concatenate([indices[:start], indices[stop:]])
            yield train_idx, test_idx
            current = stop

# Ejemplo de uso
X_demo = np.arange(20).reshape(-1, 1)
y_demo = np.arange(20)

kf = KFoldCV(n_splits=5, shuffle=True, random_state=42)

print("📊 Ejemplo de K-Fold Cross-Validation (K=5):\n")
for fold, (train_idx, test_idx) in enumerate(kf.split(X_demo), 1):
    print(f"Fold {fold}:")
    print(f"  Train: {len(train_idx)} samples (índices: {train_idx[:5]}...)")
    print(f"  Test:  {len(test_idx)} samples (índices: {test_idx})")
    print()

<a name='3.2'></a>
### 3.2 - Clase MetricasClasificacion

Implementaremos una clase completa para calcular todas las métricas de clasificación.

In [ ]:
class MetricasClasificacion:
    """
    Clase para calcular todas las métricas de clasificación binaria.
    
    Parámetros
    ----------
    y_true : array-like
        Etiquetas verdaderas (0 o 1)
    y_pred : array-like
        Predicciones (0 o 1)
    y_proba : array-like, opcional
        Probabilidades predichas para la clase positiva
    """
    
    def __init__(self, y_true, y_pred, y_proba=None):
        self.y_true = np.array(y_true)
        self.y_pred = np.array(y_pred)
        self.y_proba = np.array(y_proba) if y_proba is not None else None
        
        # Calcular componentes de la matriz de confusión
        self.TP = np.sum((self.y_true == 1) & (self.y_pred == 1))
        self.TN = np.sum((self.y_true == 0) & (self.y_pred == 0))
        self.FP = np.sum((self.y_true == 0) & (self.y_pred == 1))
        self.FN = np.sum((self.y_true == 1) & (self.y_pred == 0))
    
    def accuracy(self):
        """Calcula Accuracy"""
        return (self.TP + self.TN) / (self.TP + self.TN + self.FP + self.FN)
    
    def precision(self):
        """Calcula Precision"""
        return self.TP / (self.TP + self.FP) if (self.TP + self.FP) > 0 else 0.0
    
    def recall(self):
        """Calcula Recall (Sensitivity, TPR)"""
        return self.TP / (self.TP + self.FN) if (self.TP + self.FN) > 0 else 0.0
    
    def f1_score(self):
        """Calcula F1-Score"""
        p = self.precision()
        r = self.recall()
        return 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0
    
    def specificity(self):
        """Calcula Specificity (TNR)"""
        return self.TN / (self.TN + self.FP) if (self.TN + self.FP) > 0 else 0.0
    
    def confusion_matrix(self):
        """Retorna la matriz de confusión 2x2"""
        return np.array([[self.TN, self.FP], 
                        [self.FN, self.TP]])
    
    def classification_report(self):
        """Imprime reporte completo de métricas"""
        print("="*60)
        print(" "*15 + "REPORTE DE CLASIFICACIÓN")
        print("="*60)
        print(f"Accuracy:     {self.accuracy():.4f}")
        print(f"Precision:    {self.precision():.4f}")
        print(f"Recall:       {self.recall():.4f}")
        print(f"F1-Score:     {self.f1_score():.4f}")
        print(f"Specificity:  {self.specificity():.4f}")
        print("\nMatriz de Confusión:")
        print("                Pred Neg    Pred Pos")
        cm = self.confusion_matrix()
        print(f"True Neg        {cm[0,0]:6d}      {cm[0,1]:6d}")
        print(f"True Pos        {cm[1,0]:6d}      {cm[1,1]:6d}")
        print("="*60)

# Ejemplo de uso
y_true = np.array([0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0])
y_pred = np.array([0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1])

metricas = MetricasClasificacion(y_true, y_pred)
metricas.classification_report()

<a name='4'></a>
## 4 - Ejercicios GRADED

Ahora es tu turno de implementar las funciones clave para evaluación de modelos.

<a name='ex01'></a>
### Ejercicio 1 - Calcular Métricas de Clasificación

Implementa la función `compute_classification_metrics` que calcula Precision, Recall y F1-Score a partir de las etiquetas verdaderas y predichas.

**Instrucciones:**
1. Calcula TP, TN, FP, FN a partir de `y_true` y `y_pred`
2. Calcula Precision = $\frac{TP}{TP + FP}$
3. Calcula Recall = $\frac{TP}{TP + FN}$
4. Calcula F1-Score = $\frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$
5. Maneja divisiones por cero retornando 0.0

**Hint:** Usa operadores lógicos `&` (and) y comparaciones con NumPy.

In [ ]:
# GRADED FUNCTION: compute_classification_metrics

def compute_classification_metrics(y_true, y_pred):
    """
    Calcula Precision, Recall y F1-Score para clasificación binaria.
    
    Parámetros
    ----------
    y_true : ndarray
        Etiquetas verdaderas (0 o 1) de forma (m,)
    y_pred : ndarray
        Predicciones (0 o 1) de forma (m,)
    
    Retorna
    -------
    tuple
        (precision, recall, f1_score) - Cada uno es un float entre 0 y 1
    """
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    ### YOUR CODE STARTS HERE ###
    # Calcular componentes de la matriz de confusión
    TP = None  # True Positives: np.sum((y_true == 1) & (y_pred == 1))
    FP = None  # False Positives: np.sum((y_true == 0) & (y_pred == 1))
    FN = None  # False Negatives: np.sum((y_true == 1) & (y_pred == 0))
    
    # Calcular Precision
    precision = None  # TP / (TP + FP) si (TP + FP) > 0 sino 0.0
    
    # Calcular Recall
    recall = None  # TP / (TP + FN) si (TP + FN) > 0 sino 0.0
    
    # Calcular F1-Score
    f1_score = None  # 2 * (precision * recall) / (precision + recall) si (precision + recall) > 0 sino 0.0
    ### YOUR CODE ENDS HERE ###
    
    return precision, recall, f1_score

In [ ]:
# Test para Ejercicio 1
from tests.evaluacion.test_01_metricas_evaluacion import test_ejercicio_1_metrics

verificar_metrics = test_ejercicio_1_metrics()
verificar_metrics(compute_classification_metrics)

**Salida esperada:**
```
✅ Test 1 aprobado: Caso básico correcto
✅ Test 2 aprobado: Precision correcta
✅ Test 3 aprobado: Recall correcto
✅ Test 4 aprobado: F1-Score correcto
✅ Test 5 aprobado: Manejo de división por cero
✅ Test 6 aprobado: Predicciones perfectas

🎉 ¡Todos los tests pasaron! Ejercicio 1 completado exitosamente.
```

<a name='ex02'></a>
### Ejercicio 2 - Calcular AUC-ROC

Implementa la función `compute_auc` que calcula el área bajo la curva ROC usando la regla del trapecio.

**Instrucciones:**
1. Ordena `y_proba` y `y_true` por probabilidades decrecientes
2. Para cada threshold implícito, calcula TPR y FPR:
   - TPR = True Positives acumulados / Total Positivos
   - FPR = False Positives acumulados / Total Negativos
3. Calcula el área usando la regla del trapecio: `np.trapz(tpr, fpr)`

**Hint:** Ordena descendentemente con `np.argsort(y_proba)[::-1]`

In [ ]:
# GRADED FUNCTION: compute_auc

def compute_auc(y_true, y_proba):
    """
    Calcula el área bajo la curva ROC (AUC-ROC).
    
    Parámetros
    ----------
    y_true : ndarray
        Etiquetas verdaderas (0 o 1) de forma (m,)
    y_proba : ndarray
        Probabilidades predichas para la clase positiva de forma (m,)
    
    Retorna
    -------
    float
        Valor de AUC entre 0 y 1
    """
    
    y_true = np.array(y_true)
    y_proba = np.array(y_proba)
    
    ### YOUR CODE STARTS HERE ###
    # Ordenar por probabilidad descendente
    desc_indices = None  # np.argsort(y_proba)[::-1]
    y_true_sorted = None  # y_true[desc_indices]
    
    # Calcular número de positivos y negativos
    n_pos = None  # np.sum(y_true_sorted == 1)
    n_neg = None  # np.sum(y_true_sorted == 0)
    
    # Inicializar listas para FPR y TPR
    fpr_list = [0.0]
    tpr_list = [0.0]
    
    tp = 0
    fp = 0
    
    # Calcular TPR y FPR para cada threshold implícito
    for i in range(len(y_true_sorted)):
        if y_true_sorted[i] == 1:
            tp += 1
        else:
            fp += 1
        
        # Calcular TPR y FPR actuales
        tpr = None  # tp / n_pos si n_pos > 0 sino 0
        fpr = None  # fp / n_neg si n_neg > 0 sino 0
        
        tpr_list.append(tpr)
        fpr_list.append(fpr)
    
    # Convertir a arrays
    fpr_array = np.array(fpr_list)
    tpr_array = np.array(tpr_list)
    
    # Calcular AUC usando regla del trapecio
    auc = None  # np.trapz(tpr_array, fpr_array)
    ### YOUR CODE ENDS HERE ###
    
    return auc

In [ ]:
# Test para Ejercicio 2
from tests.evaluacion.test_01_metricas_evaluacion import test_ejercicio_2_auc

verificar_auc = test_ejercicio_2_auc()
verificar_auc(compute_auc)

**Salida esperada:**
```
✅ Test 1 aprobado: Clasificador perfecto (AUC = 1.0)
✅ Test 2 aprobado: Clasificador aleatorio (AUC ≈ 0.5)
✅ Test 3 aprobado: AUC con caso típico
✅ Test 4 aprobado: AUC está entre 0 y 1
✅ Test 5 aprobado: AUC con diferentes probabilidades

🎉 ¡Todos los tests pasaron! Ejercicio 2 completado exitosamente.
```

<a name='5'></a>
## 5 - Aplicación Práctica

<a name='5.1'></a>
### 5.1 - Evaluación Completa de un Modelo

Vamos a aplicar todas las métricas a un modelo real.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Generar dataset sintético
X, y = make_classification(n_samples=500, n_features=10, n_informative=8, 
                            n_redundant=2, random_state=42, class_sep=0.8)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Entrenar modelo
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Evaluación completa
metricas = MetricasClasificacion(y_test, y_pred, y_proba)
metricas.classification_report()

# Calcular AUC usando nuestra implementación
auc = compute_auc(y_test, y_proba)
print(f"\nAUC-ROC: {auc:.4f}")

### Visualizar Curva ROC

In [ ]:
def plot_roc_curve(y_true, y_proba):
    """Visualiza la curva ROC"""
    # Calcular FPR y TPR
    desc_indices = np.argsort(y_proba)[::-1]
    y_true_sorted = y_true[desc_indices]
    
    n_pos = np.sum(y_true_sorted == 1)
    n_neg = np.sum(y_true_sorted == 0)
    
    fpr_list = [0.0]
    tpr_list = [0.0]
    tp, fp = 0, 0
    
    for i in range(len(y_true_sorted)):
        if y_true_sorted[i] == 1:
            tp += 1
        else:
            fp += 1
        
        tpr_list.append(tp / n_pos if n_pos > 0 else 0)
        fpr_list.append(fp / n_neg if n_neg > 0 else 0)
    
    fpr = np.array(fpr_list)
    tpr = np.array(tpr_list)
    auc = np.trapz(tpr, fpr)
    
    # Graficar
    plt.figure(figsize=(10, 8))
    plt.plot(fpr, tpr, linewidth=2.5, label=f'ROC Curve (AUC = {auc:.3f})', color='#2E86AB')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier (AUC = 0.5)')
    plt.xlabel('False Positive Rate (FPR)', fontsize=12)
    plt.ylabel('True Positive Rate (TPR)', fontsize=12)
    plt.title('Curva ROC (Receiver Operating Characteristic)', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11, loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.show()

plot_roc_curve(y_test, y_proba)

<a name='5.2'></a>
### 5.2 - Comparación de Modelos con Cross-Validation

In [ ]:
def cross_val_score(model, X, y, cv=5, scoring='accuracy'):
    """
    Evalúa un modelo usando K-Fold Cross-Validation.
    
    Parámetros
    ----------
    model : objeto
        Modelo con métodos fit() y predict()
    X : array-like
        Features
    y : array-like
        Target
    cv : int
        Número de folds
    scoring : str
        Métrica: 'accuracy', 'precision', 'recall', 'f1'
    
    Retorna
    -------
    ndarray
        Scores de cada fold
    """
    kfold = KFoldCV(n_splits=cv, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, test_idx in kfold.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Entrenar y predecir
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Calcular score según métrica
        if scoring == 'accuracy':
            score = np.mean(y_pred == y_test)
        elif scoring in ['precision', 'recall', 'f1']:
            precision, recall, f1 = compute_classification_metrics(y_test, y_pred)
            score = {'precision': precision, 'recall': recall, 'f1': f1}[scoring]
        else:
            raise ValueError(f"Scoring '{scoring}' no reconocido")
        
        scores.append(score)
    
    return np.array(scores)

# Evaluar con cross-validation
print("📊 Evaluación con 5-Fold Cross-Validation:\n")

for metric in ['accuracy', 'precision', 'recall', 'f1']:
    model = LogisticRegression(random_state=42)
    scores = cross_val_score(model, X, y, cv=5, scoring=metric)
    print(f"{metric.capitalize():12s}: {scores.mean():.4f} (+/- {scores.std():.4f})")

<a name='5.3'></a>
### 5.3 - Grid Search para Hiperparámetros

Búsqueda exhaustiva de los mejores hiperparámetros.

In [ ]:
# Ejemplo de Grid Search para Logistic Regression
from sklearn.linear_model import LogisticRegression as LR

param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0],
    'penalty': ['l1', 'l2']
}

best_score = -np.inf
best_params = None
results = []

print("🔍 Grid Search en progreso...\n")

for C in param_grid['C']:
    for penalty in param_grid['penalty']:
        try:
            model = LR(C=C, penalty=penalty, solver='liblinear', random_state=42)
            scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
            mean_score = scores.mean()
            std_score = scores.std()
            
            results.append({
                'C': C,
                'penalty': penalty,
                'mean_score': mean_score,
                'std_score': std_score
            })
            
            print(f"C={C:6.3f}, penalty={penalty:2s}  →  Score: {mean_score:.4f} (+/- {std_score:.4f})")
            
            if mean_score > best_score:
                best_score = mean_score
                best_params = {'C': C, 'penalty': penalty}
        except:
            pass

print(f"\n🏆 Mejor configuración encontrada:")
print(f"   Parámetros: {best_params}")
print(f"   Score: {best_score:.4f}")

<a name='6'></a>
## 6 - Resumen y Conceptos Clave

### ✅ Conceptos Fundamentales

1. **Generalización**: El objetivo es rendimiento en datos nuevos, no solo en el training set
2. **Overfitting vs Underfitting**: Balance entre sesgo y varianza
3. **K-Fold Cross-Validation**: Estimación robusta del rendimiento usando todos los datos
4. **Matriz de Confusión**: Desglose detallado de predicciones correctas e incorrectas
5. **Trade-offs**: Precision vs Recall, sesgo vs varianza
6. **Curva ROC**: Visualización del rendimiento para diferentes thresholds
7. **AUC**: Métrica única que resume el rendimiento del clasificador

### 🎯 Lecciones Aprendidas

- ✅ **Accuracy no es suficiente** para clases desbalanceadas
- ✅ **Precision importante** cuando el costo de falsos positivos es alto (spam, fraude)
- ✅ **Recall importante** cuando el costo de falsos negativos es alto (diagnóstico médico)
- ✅ **F1-Score balancea** Precision y Recall
- ✅ **AUC-ROC robusta** a desbalance de clases y threshold-independent
- ✅ **Cross-Validation reduce** varianza en la estimación del error

### 🚀 Técnicas Avanzadas

- **Stratified K-Fold**: Mantiene la proporción de clases en cada fold
- **Repeated K-Fold**: Repite K-Fold múltiples veces para mayor robustez
- **Leave-One-Out CV**: K = n, máximo uso de datos (costoso)
- **Nested Cross-Validation**: CV anidado para selección de modelo e hiperparámetros
- **Bootstrap**: Muestreo con reemplazo para estimación de incertidumbre
- **Calibración**: Ajustar probabilidades predichas para que sean bien calibradas

<div style="background-color: #E3F2FD; padding: 20px; border-left: 5px solid #2196F3; border-radius: 5px; margin: 20px 0;">
    <h3 style="color: #1976D2; margin-top: 0;">💡 Guía de Selección de Métricas</h3>
    <p style="margin-bottom: 10px;"><strong>¿Qué métrica usar?</strong></p>
    <ul style="margin-bottom: 10px;">
        <li><strong>Clases balanceadas</strong>: Accuracy, F1-Score</li>
        <li><strong>Clases desbalanceadas</strong>: Precision, Recall, AUC-ROC, AUC-PR</li>
        <li><strong>Minimizar falsos positivos</strong>: Precision (ej: spam, recomendaciones)</li>
        <li><strong>Minimizar falsos negativos</strong>: Recall (ej: detección de cáncer, fraude)</li>
        <li><strong>Balance general</strong>: F1-Score</li>
        <li><strong>Comparar modelos sin threshold</strong>: AUC-ROC</li>
    </ul>
    <p style="margin-bottom: 0;"><strong>Buenas prácticas:</strong></p>
    <ul style="margin-bottom: 0;">
        <li>✅ Siempre usar Cross-Validation para modelos finales</li>
        <li>✅ Reportar múltiples métricas, no solo una</li>
        <li>✅ Visualizar matriz de confusión para entender errores</li>
        <li>✅ Considerar el contexto del problema al elegir métricas</li>
        <li>✅ Validar en datos completamente separados (hold-out test set)</li>
    </ul>
</div>

### 🎓 ¡Felicidades!

Has dominado las **técnicas de evaluación y validación** esenciales en Machine Learning. Ahora puedes evaluar modelos de manera rigurosa, comparar algoritmos objetivamente, y seleccionar hiperparámetros óptimos.

<a name='7'></a>
## 7 - Referencias

### 📚 Papers Fundamentales

1. **Stone, M. (1974)**. "Cross-validatory choice and assessment of statistical predictions." *Journal of the Royal Statistical Society: Series B*, 36(2), 111-133.
   - Fundamentos teóricos de cross-validation

2. **Kohavi, R. (1995)**. "A study of cross-validation and bootstrap for accuracy estimation and model selection." *IJCAI*, 14(2), 1137-1145.
   - Comparación empírica de técnicas de validación

3. **Fawcett, T. (2006)**. "An introduction to ROC analysis." *Pattern Recognition Letters*, 27(8), 861-874.
   - Tutorial completo sobre curvas ROC y AUC

4. **Davis, J., & Goadrich, M. (2006)**. "The relationship between Precision-Recall and ROC curves." *ICML*, 233-240.
   - Relación entre ROC y Precision-Recall

5. **Japkowicz, N., & Shah, M. (2011)**. *Evaluating Learning Algorithms: A Classification Perspective*. Cambridge University Press.
   - Libro completo sobre evaluación de modelos

### 📖 Libros Recomendados

1. **Hastie, T., Tibshirani, R., & Friedman, J. (2009)**. *The Elements of Statistical Learning*. Springer.
   - Capítulo 7: Model Assessment and Selection
   - Disponible gratis: https://hastie.su.domains/ElemStatLearn/

2. **Kuhn, M., & Johnson, K. (2013)**. *Applied Predictive Modeling*. Springer.
   - Enfoque práctico en evaluación y selección de modelos

3. **Provost, F., & Fawcett, T. (2013)**. *Data Science for Business*. O'Reilly.
   - Métricas de evaluación en contextos empresariales

### 🌐 Recursos Online

1. **Scikit-learn User Guide - Model Evaluation**
   - https://scikit-learn.org/stable/modules/model_evaluation.html

2. **Google ML Crash Course - Classification**
   - https://developers.google.com/machine-learning/crash-course/classification

3. **ROC Curves and Under the Curve (AUC) Explained**
   - https://www.youtube.com/watch?v=4jRBRDbJemM (StatQuest)

4. **Cross-Validation Explained**
   - https://www.youtube.com/watch?v=fSytzGwwBVw (StatQuest)

### 💻 Herramientas

1. **scikit-learn**: https://scikit-learn.org/ - Implementaciones optimizadas de todas las métricas
2. **yellowbrick**: https://www.scikit-yb.org/ - Visualización de métricas de evaluación
3. **imbalanced-learn**: https://imbalanced-learn.org/ - Métricas para datasets desbalanceados
4. **ROCR (R)**: https://cran.r-project.org/package=ROCR - Análisis ROC en R